# 05 — Classification and Baselines

This notebook evaluates the distance-based features generated by Notebook 04.

Models and baselines:
1. SVM with class weighting.
2. ANN with balanced training data through positive-class oversampling.
3. Pattern-similarity threshold baseline.
4. Majority-class baseline for AUPRC/accuracy context.

The test set is kept untouched. Oversampling is performed **only on the training set**.

Because the current distance-feature dataset contains one six-hour window per patient record, the split is performed at the record/patient-window level using the stored `patient_id`.


In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "outputs").exists() and (PROJECT_ROOT.parent / "outputs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

FEATURE_DIR = PROJECT_ROOT / "outputs" / "features"
MODEL_DIR = PROJECT_ROOT / "outputs" / "models"
EVAL_DIR = PROJECT_ROOT / "outputs" / "evaluation"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_FILE = FEATURE_DIR / "sequence_distance_features.csv"

RANDOM_STATE = 42
TEST_SIZE = 0.20

print("Project root:", PROJECT_ROOT)
print("Feature file:", FEATURE_FILE)


## 1. Load distance features

The classifier uses similarity features generated in Notebook 04.

`patient_id`, distance columns, and best-hour columns are retained for traceability but are not used as predictive inputs.


In [ ]:
if not FEATURE_FILE.exists():
    raise FileNotFoundError(f"Run Notebook 04 first. Missing: {FEATURE_FILE}")

features_df = pd.read_csv(FEATURE_FILE)

print("Dataset shape:", features_df.shape)
display(features_df.head())

similarity_columns = [
    c for c in features_df.columns
    if c.endswith("_similarity")
]

if not similarity_columns:
    raise ValueError("No similarity features found.")

X = features_df[similarity_columns].copy()
y = features_df["label"].astype(int).copy()

print("Similarity features:", similarity_columns)
print("\nClass counts:")
print(y.value_counts())
print("\nPositive prevalence:", y.mean())


## 2. Stratified train/test split

The split is stratified to preserve the positive/negative proportion.

The test set is never oversampled or used to select model parameters.


In [ ]:
indices = np.arange(len(features_df))

train_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTraining class counts:")
print(y_train.value_counts())

print("\nTest class counts:")
print(y_test.value_counts())


## 3. Evaluation helper

The main metrics are Precision, Recall, F1, AUROC, and AUPRC.

AUPRC is particularly important here because the positive class is much smaller than the negative class.


In [ ]:
def evaluate_model(name, y_true, y_pred, y_score):
    result = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auroc": roc_auc_score(y_true, y_score),
        "auprc": average_precision_score(y_true, y_score)
    }

    print(f"\n===== {name} =====")
    for metric in ["accuracy", "precision", "recall", "f1", "auroc", "auprc"]:
        print(f"{metric.upper():9s}: {result[metric]:.4f}")

    print("\nClassification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=["Non-sepsis", "Pre-sepsis"],
            zero_division=0
        )
    )

    return result


## 4. Majority-class baseline

This baseline always predicts the majority class.

It provides context for the accuracy and AUPRC values. It is not intended to be a useful detector.


In [ ]:
majority_class = int(y_train.mode()[0])

majority_pred = np.full(len(y_test), majority_class, dtype=int)
majority_score = np.full(
    len(y_test),
    float(y_train.mean())
)

majority_result = evaluate_model(
    "Majority baseline",
    y_test,
    majority_pred,
    majority_score
)

print("Majority class:", majority_class)
print("Training positive prevalence:", y_train.mean())


## 5. SVM with class weighting

The SVM uses the distance-derived similarity features.

`class_weight="balanced"` increases the penalty for misclassifying the minority positive class without modifying the test set.


In [ ]:
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "svm",
        SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            probability=True,
            class_weight="balanced",
            random_state=RANDOM_STATE
        )
    )
])

svm_model.fit(X_train, y_train)

svm_pred = svm_model.predict(X_test)
svm_score = svm_model.predict_proba(X_test)[:, 1]

svm_result = evaluate_model(
    "SVM",
    y_test,
    svm_pred,
    svm_score
)


## 6. Balance the ANN training data

The previous ANN version learned the majority class too strongly.

To address this, positive training examples are oversampled **only within the training set** until the two classes have equal counts.

The test set remains completely untouched.


In [ ]:
def oversample_minority(X_data, y_data, random_state=42):
    rng = np.random.default_rng(random_state)

    X_array = np.asarray(X_data)
    y_array = np.asarray(y_data)

    positive_idx = np.where(y_array == 1)[0]
    negative_idx = np.where(y_array == 0)[0]

    if len(positive_idx) == 0 or len(negative_idx) == 0:
        raise ValueError("Both classes are required for oversampling.")

    if len(positive_idx) == len(negative_idx):
        return X_array.copy(), y_array.copy()

    if len(positive_idx) < len(negative_idx):
        minority_idx = positive_idx
        majority_idx = negative_idx
        minority_label = 1
    else:
        minority_idx = negative_idx
        majority_idx = positive_idx
        minority_label = 0

    extra_count = len(majority_idx) - len(minority_idx)

    sampled_idx = rng.choice(
        minority_idx,
        size=extra_count,
        replace=True
    )

    balanced_idx = np.concatenate([
        majority_idx,
        minority_idx,
        sampled_idx
    ])

    rng.shuffle(balanced_idx)

    return X_array[balanced_idx], y_array[balanced_idx]


X_train_ann, y_train_ann = oversample_minority(
    X_train,
    y_train,
    random_state=RANDOM_STATE
)

print("Original training counts:")
print(y_train.value_counts())

print("\nBalanced ANN training counts:")
print(pd.Series(y_train_ann).value_counts())


## 7. ANN classifier

The ANN uses the same distance/similarity feature representation as the SVM.

The architecture is intentionally small:

`input → 32 → 16 → output`

Early stopping limits unnecessary training.


In [ ]:
ann_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "ann",
        MLPClassifier(
            hidden_layer_sizes=(32, 16),
            activation="relu",
            solver="adam",
            alpha=1e-4,
            learning_rate_init=1e-3,
            max_iter=200,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=15,
            random_state=RANDOM_STATE
        )
    )
])

ann_model.fit(X_train_ann, y_train_ann)

ann_pred = ann_model.predict(X_test)
ann_score = ann_model.predict_proba(X_test)[:, 1]

ann_result = evaluate_model(
    "ANN (balanced training)",
    y_test,
    ann_pred,
    ann_score
)


## 8. Pattern-similarity threshold baseline

This is a simple interpretable baseline based on the maximum similarity to any discriminative pattern.

The threshold is learned from the training data only.

It is deliberately kept separate from the SVM/ANN models.


In [ ]:
threshold_feature = "max_pattern_similarity"

if threshold_feature not in X_train.columns:
    raise ValueError(f"Missing {threshold_feature}")

# Use the median positive training similarity as the threshold.
baseline_threshold = float(
    X_train.loc[y_train == 1, threshold_feature].median()
)

baseline_score = X_test[threshold_feature].to_numpy()
baseline_pred = (baseline_score >= baseline_threshold).astype(int)

baseline_result = evaluate_model(
    f"Pattern similarity threshold ({baseline_threshold:.3f})",
    y_test,
    baseline_pred,
    baseline_score
)

print("Threshold learned from positive training windows:", baseline_threshold)


## 9. Optional qSOFA-style available-feature baseline

The original project proposal mentions comparison with threshold-based/qSOFA rules.

The current saved feature matrix contains **discretized symbolic states**, not the original raw vital measurements and does not contain a Glasgow Coma Scale/altered-mentation variable. Therefore a true clinical qSOFA score cannot be reconstructed from this file without inventing missing information.

Instead, this notebook reports the pattern-similarity threshold as the project's threshold baseline and explicitly records why a true qSOFA implementation is not included here.


In [ ]:
qsofa_note = {
    "true_qSOFA_implemented": False,
    "reason": (
        "The current feature file contains discretized symbolic states and "
        "does not provide the original raw measurements plus a Glasgow Coma "
        "Scale/altered-mentation variable required to reproduce qSOFA faithfully."
    ),
    "threshold_baseline_used": "Maximum discriminative-pattern similarity"
}

with open(EVAL_DIR / "qsofa_baseline_note.json", "w", encoding="utf-8") as f:
    json.dump(qsofa_note, f, indent=2)

print(qsofa_note["reason"])


## 10. Compare all baseline/model results


In [ ]:
results_df = pd.DataFrame([
    majority_result,
    svm_result,
    ann_result,
    baseline_result
])

display(
    results_df[
        [
            "model",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "auroc",
            "auprc"
        ]
    ].sort_values("auprc", ascending=False)
)


## 11. Confusion matrices

Confusion matrices show how many positive and negative windows are correctly or incorrectly classified.


In [ ]:
model_predictions = {
    "SVM": svm_pred,
    "ANN balanced": ann_pred,
    "Pattern threshold": baseline_pred
}

for name, predictions in model_predictions.items():
    cm = confusion_matrix(y_test, predictions)

    plt.figure(figsize=(5, 4))
    plt.imshow(cm)
    plt.title(f"Confusion Matrix — {name}")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.xticks([0, 1], ["Non-sepsis", "Pre-sepsis"])
    plt.yticks([0, 1], ["Non-sepsis", "Pre-sepsis"])

    for i in range(2):
        for j in range(2):
            plt.text(j, i, cm[i, j], ha="center", va="center")

    plt.colorbar()
    plt.tight_layout()

    safe_name = (
        name.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    plt.savefig(
        EVAL_DIR / f"confusion_matrix_{safe_name}.png",
        dpi=200,
        bbox_inches="tight"
    )
    plt.show()


## 12. Save models, results, and split information


In [ ]:
joblib.dump(
    svm_model,
    MODEL_DIR / "svm_model.joblib"
)

joblib.dump(
    ann_model,
    MODEL_DIR / "ann_model_balanced.joblib"
)

results_df.to_csv(
    EVAL_DIR / "classification_results.csv",
    index=False
)

split_information = {
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "train_indices": train_idx.tolist(),
    "test_indices": test_idx.tolist(),
    "feature_columns": similarity_columns,
    "train_positive_count": int((y_train == 1).sum()),
    "train_negative_count": int((y_train == 0).sum()),
    "test_positive_count": int((y_test == 1).sum()),
    "test_negative_count": int((y_test == 0).sum())
}

with open(
    EVAL_DIR / "classification_split.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(split_information, f, indent=2)

with open(
    MODEL_DIR / "baseline_threshold.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(str(baseline_threshold))

print("Saved:")
print(" -", MODEL_DIR / "svm_model.joblib")
print(" -", MODEL_DIR / "ann_model_balanced.joblib")
print(" -", EVAL_DIR / "classification_results.csv")
print(" -", EVAL_DIR / "classification_split.json")
print(" -", MODEL_DIR / "baseline_threshold.txt")
print(" -", EVAL_DIR / "qsofa_baseline_note.json")


## 13. Final sanity checks


In [ ]:
assert len(results_df) == 4
assert np.isfinite(
    results_df[["auroc", "auprc"]].to_numpy()
).all()

assert set(svm_pred).issubset({0, 1})
assert set(ann_pred).issubset({0, 1})
assert set(baseline_pred).issubset({0, 1})

assert len(y_train_ann) == len(X_train_ann)
assert set(np.unique(y_train_ann)) == {0, 1}

print("All classification sanity checks passed.")
print("Notebook 05 is ready for Notebook 06.")
